# Fine-tune the plate detector on Pakistani plates

Runs on a free Colab GPU. Trains a single-class license-plate detector to
replace `models/plate_detector.pt`.

**Why this exists.** The shipped detector is `Koushim/yolov8-license-plate-detection`
— a pretrained YOLOv8n from HuggingFace, trained on non-local plates and never
validated against Pakistani ones. A sweep over real gate frames showed it needs a
vehicle ~300px wide before it yields an OCR-able plate, and finds nothing at all on
tall/narrow motorcycle-shaped crops. See `docs/DECISIONS.md` decisions 1-3.

**Runtime > Change runtime type > T4 GPU** before running anything.


## 1. Setup


In [ ]:
!pip install -q ultralytics roboflow
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


## 2. Get the dataset

API key: Roboflow **Settings > API Keys** (free account).

Datasets chosen in `docs/DECISIONS.md` decision 2:
- `license-plate-yf8bv/pakistan-license-plate-detection` (~6,161 images) — primary
- `malik-kashif-saeed-aswwf/pakistani-number-plates` (336) — optional supplement

If the API is awkward, use **Download Dataset > YOLOv8 > show download code** on the
dataset page and paste that snippet here instead — same resulting layout.


In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass('Roboflow API key: ')

WORKSPACE = 'license-plate-yf8bv'
PROJECT   = 'pakistan-license-plate-detection'
VERSION   = 1          # bump if the dataset page shows a newer version

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download('yolov8')
DATA_YAML = dataset.location + '/data.yaml'
print('data.yaml ->', DATA_YAML)


### Sanity-check the dataset first

Worth 30 seconds. A multi-class dataset, or one whose single class is not the plate,
trains happily and produces a useless model.


In [ ]:
import yaml, glob, os

cfg = yaml.safe_load(open(DATA_YAML))
print('classes:', cfg.get('names'), '| nc =', cfg.get('nc'))
for split in ('train', 'valid', 'test'):
    d = os.path.join(dataset.location, split, 'images')
    print(split.ljust(6), len(glob.glob(d + '/*')) if os.path.isdir(d) else 'absent')

if cfg.get('nc') != 1:
    raise SystemExit(
        'Expected a single plate class, got nc=%s: %s. A character-level dataset '
        '(0-9, A-Z) trains a different thing entirely -- see DECISIONS.md decision 5.'
        % (cfg.get('nc'), cfg.get('names'))
    )


## 3. Train

`imgsz=960` deliberately: the pipeline runs this model on **vehicle crops**, not whole
frames, and plate boxes in those crops are small (20-90px wide on real footage).
Training at 640 teaches it on plates smaller than it meets at inference.

Base model `yolo26n` per decision 4 (better small-object performance), falling back to
`yolov8n` — matching the currently shipped model — if those weights are unavailable.


In [ ]:
from ultralytics import YOLO

BASE = 'yolo26n.pt'
try:
    model = YOLO(BASE)
except Exception as e:
    print('%s unavailable (%s); falling back to yolov8n.pt' % (BASE, e))
    BASE = 'yolov8n.pt'
    model = YOLO(BASE)
print('base:', BASE)

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=960,
    batch=16,          # drop to 8 if you hit CUDA OOM on a T4
    patience=20,       # early-stop when val stops improving
    project='runs', name='plate_finetune',
    pretrained=True,
)


## 4. Validate


In [ ]:
metrics = model.val()
print('mAP50     %.4f' % metrics.box.map50)
print('mAP50-95  %.4f' % metrics.box.map)
print('precision %.4f   recall %.4f' % (metrics.box.mp, metrics.box.mr))
print()
print('Recall is the number that matters here: the pipeline fails by finding NO')
print('plate, not by finding wrong ones -- the plate-format regex rejects those.')


## 5. Check it on YOUR footage

**Do not skip this.** The dataset is someone else's camera geometry. Your gate cameras
have their own height, angle and distance, and that mismatch is exactly what makes the
current model fail. A good mAP on Roboflow's val split does not mean it works here.

Upload a few frames from your clips — `output/*_evidence/*_frame.jpg` from a previous
run works well.


In [ ]:
from google.colab import files
from pathlib import Path

Path('mytest').mkdir(exist_ok=True)
uploaded = files.upload()          # pick frames/crops from your own footage
for name in uploaded:
    Path('mytest', name).write_bytes(uploaded[name])

for img in sorted(Path('mytest').iterdir()):
    r = model.predict(str(img), conf=0.25, imgsz=960, verbose=False)[0]
    n = 0 if r.boxes is None else len(r.boxes)
    sizes = []
    if n:
        for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            sizes.append('%dx%d@%.2f' % (b[2]-b[0], b[3]-b[1], c))
    print('%s: %d plate(s) %s' % (img.name, n, sizes))
    r.save(filename='pred_' + img.name)


### Compare against the model you are replacing

A number on its own means nothing. Upload the current `models/plate_detector.pt` and
run both over the same images. If the fine-tune does not win on **your** frames, it is
not an improvement regardless of its mAP.


In [ ]:
shipped_up = files.upload()        # upload models/plate_detector.pt
shipped = YOLO(list(shipped_up)[0])

print('%-32s %10s %10s' % ('image', 'shipped', 'finetuned'))
for img in sorted(Path('mytest').iterdir()):
    a = shipped.predict(str(img), conf=0.25, imgsz=960, verbose=False)[0]
    b = model.predict(str(img),   conf=0.25, imgsz=960, verbose=False)[0]
    na = 0 if a.boxes is None else len(a.boxes)
    nb = 0 if b.boxes is None else len(b.boxes)
    print('%-32s %10d %10d' % (img.name[:32], na, nb))


## 6. Export and wire it in

Download `best.pt`, then in the repo:

```bash
# keep the old one so you can A/B and roll back
mv models/plate_detector.pt models/plate_detector_koushim.pt
cp ~/Downloads/best.pt models/plate_detector.pt

python run_pipeline.py --video sample_data/dataset_clear_01.mp4 \
  --events-json output/dataset_clear_01_events.json \
  --evidence-dir output/dataset_clear_01_evidence \
  --plate-imgsz 960

python scripts/score_accuracy.py --report docs/ACCURACY_REPORT_FINETUNED.md
```

Pass `--plate-imgsz 960` to match what you trained at — the pipeline otherwise leaves
ultralytics at its 640 default.

**Change one thing at a time.** Swap the plate model, re-measure, *then* consider the
YOLO26 vehicle-detector upgrade. Doing both at once makes the result uninterpretable —
which is the whole reason tracker fixes were deferred (decision 8).

Check the run's rejection breakdown afterwards: if `no_plate_detected` drops but
`plate_box_too_small` rises, the new model is finding plates the geometry filters then
discard — tune `--plate-min-width/height` rather than retraining again.


In [ ]:
from google.colab import files
import os
best = 'runs/plate_finetune/weights/best.pt'
print('size:', round(os.path.getsize(best) / 1e6, 1), 'MB')
files.download(best)
